# Retrieval Comparison

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


from src.rag.config import load_config

from src.rag.embedding.factory import EmbeddingFactory
from src.rag.preprocessing.processor import TextProcessor

from src.rag.vector_store.faiss import FAISSVectorStore

from src.rag.retrieval.bm25 import BM25Retriever
from src.rag.retrieval.embedding import EmbeddingRetriever
from src.rag.retrieval.hybrid import HybridRetriever


In [ ]:
config = load_config("../configs/rag.yaml")

config


In [ ]:
processor = TextProcessor()


embedding_model = EmbeddingFactory.create(
    provider=config["embedding"]["provider"],
    model_name=config["embedding"]["model"]
)


In [ ]:
# Load FAISS index

faiss_path = Path(
    "../data/indexes/product_comments"
)

vector_store = FAISSVectorStore()

vector_store.load(
    faiss_path
)

vector_store.documents.head()


In [ ]:
# Load BM25 index

bm25_path = Path(
    "../data/indexes/product_comments_bm25"
)

bm25_retriever = BM25Retriever(
    processor=processor
)

bm25_retriever.load(
    bm25_path
)


In [ ]:
# Create embedding retriever from loaded FAISS

embedding_retriever = EmbeddingRetriever(
    documents=vector_store.documents,
    embedding_model=embedding_model,
    processor=processor
)

embedding_retriever.vector_store = vector_store


In [ ]:
# Create hybrid retriever

hybrid_config = config["retrieval"]["hybrid"]

hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25_retriever,
    embedding_retriever=embedding_retriever,
    bm25_weight=hybrid_config["bm25_weight"],
    embedding_weight=hybrid_config["embedding_weight"]
)


In [ ]:
query = "آیا برای پوست چرب مناسب است؟"

top_k = 5


## BM25

In [ ]:
bm25_results = bm25_retriever.retrieve(
    query,
    top_k=top_k
)

bm25_results[
    ["body", "rate", "score"]
]


## Embedding

In [ ]:
embedding_results = embedding_retriever.retrieve(
    query,
    top_k=top_k
)

embedding_results[
    ["body", "rate", "score"]
]


## Hybrid

In [ ]:
hybrid_results = hybrid_retriever.retrieve(
    query,
    top_k=top_k
)

hybrid_results


## Comparison

مقایسه خروجی سه روش:

- BM25
- Embedding
- Hybrid
